# Rangos laterales — Filtro 1

**Este fichero es el CÓDIGO, no el gráfico.** Al ejecutarlo genera dos
ficheros en esta misma carpeta:

- `notebooks/rangos_btc.html`
- `notebooks/rangos_ondo.html`

**Esos son los gráficos.** Ábrelos con Chrome (clic derecho → Abrir
con → Chrome). En este equipo los `.html` están asociados a Internet
Explorer, que no sabe dibujar Plotly y muestra código en bruto.

---

Carga las velas de 4h desde la caché en `data/raw/` (sin descargar de
Kraken), detecta los rangos laterales (SPEC.md §5) y traza sobre cada
uno su perfil de volumen (SPEC.md §4).

### Cómo leer el gráfico

Cada rango es un rectángulo relleno, con el color de su ventana de
detección. **Los dos tipos son operables**; el nombre indica la
escala:

- **Principal** (ventanas de 150-400 velas, 1-3 meses) — relleno
  tenue. Laterales de estructura mayor y los más operables: cuanto
  más tiempo pasa el precio construyendo el rango, más volumen
  acumula el perfil y más fiables son sus niveles.
- **Secundario** (40-60 velas, 1-2 semanas) — relleno más intenso.
  Rangos anidados dentro de los principales.

La **opacidad indica la calidad**: cuanto más nítido, más lateral y
limpio es el rango.

Sobre cada rango, su **FRVP**: el histograma de volumen al arranque y
los tres niveles operativos —**POC** en línea continua, **VAH** y
**VAL** discontinuas— proyectados hacia la derecha en los rangos
principales. Ahí es donde se abre posición cuando el precio vuelve a
testearlos.

### Cambiar la vista sin salir del navegador

El gráfico lleva un **menú desplegable arriba a la izquierda** con
ocho vistas: completa, solo principales, solo secundarios, sin FRVP,
todos los rangos detectados, y tres de operaciones (todas, ganadoras
y perdedoras). Cambian al instante, sin volver a ejecutar nada.

Las tres últimas dibujan las operaciones del experimento de toques
del FRVP (SPEC.md §8) y solo aparecen si el experimento ya se ha
corrido:

    .venv\Scripts\python.exe experiments\exp_toques_frvp.py

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Añade la raíz del proyecto a sys.path para poder importar `data.loader`
# tanto si Jupyter arranca en `notebooks/` como si arranca en la raíz.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.frvp import calcular_frvp, timeframe_construccion
from core.imbalances import detectar_imbalances, evolucion_imbalance
from core.levels import construir_niveles
from core.osciladores import adx, fase_ttm, momento_ttm, squeeze
from core.range_detector import detectar_rangos_laterales, seleccionar_rangos
from data.loader import cargar_config, descargar_ohlcv

## Parámetros

In [2]:
config = cargar_config(PROJECT_ROOT / "config.yaml")

SIMBOLOS: list[str] = config["datos"]["simbolos"]
TIMEFRAME = "4h"
HISTORICO_ANIOS: float = config["datos"]["historico_anios"]
VENTANAS = [v["velas"] for v in config["filtro_1_rango_lateral"]["ventanas"]]

# Timeframe en el que se buscan los imbalances (SPEC.md §9).
TIMEFRAME_IMBALANCES: str = config["imbalances"]["timeframe"]

# Rango de fechas a visualizar (UTC). Usa None para no acotar.
# Acota los DATOS, no la vista: para cambiar qué se muestra está el
# menú desplegable dentro del propio gráfico.
FECHA_INICIO: str | None = None  # p. ej. "2025-11-01"
FECHA_FIN: str | None = None  # p. ej. "2026-09-01"

print("Símbolos:", ", ".join(SIMBOLOS))
print("Ventanas (velas de 4h):", VENTANAS)
print("Imbalances:", TIMEFRAME_IMBALANCES)

Símbolos: ONDO/USD:USD, BTC/USD:USD
Ventanas (velas de 4h): [40, 60, 150, 250, 400]
Imbalances: 1w


## Carga de datos desde caché

`forzar=False`: si el parquet ya existe en `data/raw/` se lee de ahí y
no se contacta con el exchange.

In [3]:
velas: dict[str, pd.DataFrame] = {
    symbol: descargar_ohlcv(symbol, TIMEFRAME, HISTORICO_ANIOS, forzar=False)
    for symbol in SIMBOLOS
}

# El FRVP se construye con velas más finas que las de decisión cuando
# el rango es corto (SPEC.md §1), para reducir el error de asignación
# intra-vela. Se cargan también esas series.
velas_finas: dict[str, dict[str, pd.DataFrame]] = {
    symbol: {
        tf: (velas[symbol] if tf == TIMEFRAME
             else descargar_ohlcv(symbol, tf, HISTORICO_ANIOS, forzar=False))
        for tf in ("15m", "1h", "4h")
    }
    for symbol in SIMBOLOS
}

# Los imbalances se buscan en el SEMANAL: son zonas de estructura
# mayor, y en 4h saldrían decenas sin peso ninguno.
velas_semanales: dict[str, pd.DataFrame] = {
    symbol: descargar_ohlcv(symbol, TIMEFRAME_IMBALANCES, HISTORICO_ANIOS,
                            forzar=False)
    for symbol in SIMBOLOS
}

for symbol, df in velas.items():
    print(f"{symbol}: {len(df)} velas de {TIMEFRAME}, "
          f"{len(velas_semanales[symbol])} de {TIMEFRAME_IMBALANCES}")

ONDO/USD:USD: 4379 velas de 4h, 103 de 1w
BTC/USD:USD: 4379 velas de 4h, 103 de 1w


## Estilo — tema desierto

Fondo arena, velas en marrón muy oscuro (huecas al alza, rellenas a
la baja) y rangos en tonos terrosos. Los rectángulos van rellenos y
sin contorno.

Un fondo claro y cálido cansa menos la vista en sesiones largas que
el negro puro, y sirve igual para pantalla que para imprimir en la
memoria del TFM: un gráfico de fondo negro en papel queda mal y se
come el tóner.

Sin borde, el tipo de rango se distingue por la intensidad del
relleno: el principal va más tenue porque es más grande y se
solaparía con los secundarios que lleva dentro.

Los grosores se mantienen finos a propósito. En un gráfico con
decenas de rangos y sus tres niveles cada uno, una línea de más de un
punto satura la vista: el precio deja de leerse y los niveles se
confunden entre sí.

In [4]:
# Paleta del gráfico: tema DESIERTO.
#
# Para volver al tema oscuro tipo TradingView, sustituir este bloque
# por:  FONDO "#131722", REJILLA "#1c202b", TEXTO "#a9b1bd",
#       TEXTO_TENUE "#6b7280", VELA "#d1d4dc", y la paleta
#       ["#5b8ff9", "#2ec7a9", "#f2637b", "#b07aff", "#f0a13c"].
FONDO = "#f3ece0"        # arena
PANEL = "#ede4d5"        # arena algo más tostada, para el menú
REJILLA = "#e2d7c3"      # trazo de rejilla, apenas visible
TEXTO = "#4f463a"        # marrón oscuro
TEXTO_TENUE = "#8d8171"  # marrón grisáceo
VELA = "#33291d"         # casi negro cálido

# Tipografía del sistema. Nada de fuentes web: el HTML tiene que
# abrirse sin conexión.
FUENTE = "Segoe UI, -apple-system, Roboto, Helvetica, Arial, sans-serif"

# Un color por tamaño de ventana, de más corta a más larga. Tonos
# terrosos: sobre fondo claro un color saturado chilla, y con cinco
# ventanas en pantalla a la vez el gráfico se vuelve ilegible.
PALETA = [
    "#3b5b8c",  # 40  — índigo
    "#4f7a4a",  # 60  — salvia
    "#b0522c",  # 150 — terracota
    "#7a4b73",  # 250 — ciruela
    "#a37518",  # 400 — ocre
]
COLOR_POR_VENTANA = {n: PALETA[i % len(PALETA)] for i, n in enumerate(VENTANAS)}
COLOR_POR_DEFECTO = "#8d8171"

# El tipo de rango solo cambia la intensidad del relleno: sin contorno,
# es lo único que los separa visualmente. Sobre fondo claro el relleno
# pesa más que sobre oscuro, así que las opacidades bajan.
ESTILO_POR_TIPO = {
    "principal": {"relleno": 0.11},
    "secundario": {"relleno": 0.20},
}

# Grosores de las líneas del FRVP. Finos a propósito (ver arriba).
GROSOR_POC = 1.1
GROSOR_VA = 0.8
GROSOR_VELA = 0.9

# Por debajo de esta nota el rectángulo se dibuja muy tenue.
CALIDAD_DESTACADA = 0.60


def rgba(color_hex, opacidad):
    """Convierte un color hexadecimal en ``rgba(...)`` de Plotly.

    Parameters
    ----------
    color_hex : str
        Color en formato ``"#rrggbb"``.
    opacidad : float
        Canal alfa, entre 0 y 1.

    Returns
    -------
    str
        El color en formato ``"rgba(r, g, b, a)"``.
    """
    crudo = color_hex.lstrip("#")
    r, g, b = (int(crudo[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {max(0.0, min(1.0, opacidad)):.3f})"


def visibilidad(calidad):
    """Traduce la nota de calidad a un factor de opacidad.

    Los rangos flojos no se ocultan —cumplen las reglas y pueden
    interesar— pero se apagan para que no compitan visualmente con los
    buenos.

    Parameters
    ----------
    calidad : float
        Nota entre 0 y 1.

    Returns
    -------
    float
        Factor entre 0.4 y 1.0.
    """
    if calidad >= CALIDAD_DESTACADA:
        return 1.0
    return 0.4 + 0.6 * (calidad / CALIDAD_DESTACADA)

## Perfil de volumen (FRVP)

Sobre cada rango se traza su perfil y se extraen los tres niveles
operativos (SPEC.md §4): **POC** (el precio con más volumen) y
**VAH/VAL** (los bordes de la zona que concentra el 70%).

El perfil se dibuja como histograma horizontal en el arranque del
rango, y los niveles se proyectan hacia la derecha en los rangos
principales: ahí es donde se abre posición cuando el precio vuelve a
testearlos, largo si llega desde abajo y corto si llega desde arriba.

Cada perfil hereda el color de su rango, así se ve de un vistazo qué
niveles vienen de qué caja.

In [5]:
# Anchura del histograma del perfil, como fracción del ancho del rango.
ANCHO_PERFIL = 0.16
# Bins del histograma al DIBUJAR. El cálculo usa los 1000 de SPEC.md
# §4; para pintar se agregan, porque 1000 puntos por rango no se
# distinguen en pantalla y multiplican el peso del HTML.
BINS_DIBUJO = 90


def agregar_perfil(precios, volumen, n_bins=BINS_DIBUJO):
    """Agrupa el perfil en menos bins, solo para dibujarlo.

    Parameters
    ----------
    precios, volumen : np.ndarray
        Centro y volumen de cada bin del perfil calculado.
    n_bins : int
        Bins de salida.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Precios y volúmenes agregados.
    """
    if len(precios) <= n_bins:
        return precios, volumen
    grupo = np.linspace(0, n_bins, len(precios), endpoint=False).astype(int)
    p = np.bincount(grupo, weights=precios) / np.bincount(grupo)
    v = np.bincount(grupo, weights=volumen)
    return p, v


def perfil_de(symbol, rango):
    """Calcula el FRVP de un rango con la granularidad que le toca.

    Parameters
    ----------
    symbol : str
        Símbolo unificado de CCXT.
    rango : namedtuple
        Fila del DataFrame de rangos.

    Returns
    -------
    dict | None
        Salida de `calcular_frvp`, o None si el tramo no tiene volumen.
    """
    n_velas = len(velas[symbol].loc[rango.inicio:rango.fin])
    tf = timeframe_construccion(n_velas, config)
    return calcular_frvp(velas_finas[symbol][tf], rango.inicio, rango.fin, config)


def forma_caja(rango, color, vis):
    """Rectángulo del rango: relleno sin contorno, como en TradingView.

    Parameters
    ----------
    rango : namedtuple
        Fila del DataFrame de rangos.
    color : str
        Color hexadecimal.
    vis : float
        Factor de opacidad por calidad.

    Returns
    -------
    dict
        Forma de Plotly.
    """
    estilo = ESTILO_POR_TIPO[rango.tipo]
    return {
        "type": "rect", "xref": "x", "yref": "y",
        "x0": rango.inicio, "x1": rango.fin,
        "y0": rango.suelo, "y1": rango.techo,
        "line": {"width": 0},
        "fillcolor": rgba(color, estilo["relleno"] * vis),
        "layer": "below",
    }


def formas_niveles(rango, perfil, color, vis, x_fin):
    """Las tres líneas del FRVP: VAL, POC y VAH.

    Parameters
    ----------
    rango : namedtuple
        Fila del DataFrame de rangos.
    perfil : dict
        Salida de `calcular_frvp`.
    color : str
        Color hexadecimal.
    vis : float
        Factor de opacidad.
    x_fin : pd.Timestamp
        Hasta dónde llega la línea.

    Returns
    -------
    list[dict]
        Formas de Plotly.
    """
    return [
        {
            "type": "line", "xref": "x", "yref": "y",
            "x0": rango.inicio, "x1": x_fin,
            "y0": perfil[nivel], "y1": perfil[nivel],
            "line": {"color": rgba(color, vis), "width": grosor, "dash": guion},
            "layer": "below",
        }
        for nivel, guion, grosor in (
            ("val", "dot", GROSOR_VA),
            ("poc", "solid", GROSOR_POC),
            ("vah", "dot", GROSOR_VA),
        )
    ]


def etiqueta_rango(rango, color, vis):
    """Rótulo del rango, pegado a su esquina superior izquierda.

    Sin recuadro de fondo y en el color del propio rango: con decenas
    de rangos en pantalla, un fondo opaco por etiqueta tapa el precio.

    Parameters
    ----------
    rango : namedtuple
        Fila del DataFrame de rangos.
    color : str
        Color hexadecimal.
    vis : float
        Factor de opacidad por calidad.

    Returns
    -------
    dict
        Anotación de Plotly.
    """
    return {
        "x": rango.inicio, "y": rango.techo,
        "xref": "x", "yref": "y",
        "text": f"N{rango.ventana}·{rango.calidad:.2f}",
        "showarrow": False,
        "xanchor": "left", "yanchor": "bottom",
        "xshift": 2, "yshift": 1,
        "font": {"size": 8, "color": rgba(color, min(1.0, vis + 0.15)),
                 "family": FUENTE},
    }


def traza_perfil(rango, perfil, color, vis):
    """Histograma horizontal del perfil, al arranque del rango.

    Parameters
    ----------
    rango : namedtuple
        Fila del DataFrame de rangos.
    perfil : dict
        Salida de `calcular_frvp`.
    color : str
        Color hexadecimal.
    vis : float
        Factor de opacidad.

    Returns
    -------
    go.Scatter | None
        La traza, o None si el perfil no tiene volumen.
    """
    p, v = agregar_perfil(perfil["precios"], perfil["volumen"])
    if v.max() <= 0:
        return None
    ancho = (rango.fin - rango.inicio) * ANCHO_PERFIL
    x = [rango.inicio + ancho * (u / v.max()) for u in v]
    return go.Scatter(
        x=[rango.inicio] + x + [rango.inicio],
        y=[p[0]] + list(p) + [p[-1]],
        fill="toself",
        fillcolor=rgba(color, 0.20 * vis),
        line={"width": 0},
        hoverinfo="skip",
        showlegend=False,
    )

## Operaciones del experimento

Dibuja sobre el precio las operaciones que produjo
`experiments/exp_toques_frvp.py`, leídas de su CSV. **No las
recalcula**: si cambias parámetros hay que volver a correr el
experimento antes de regenerar el gráfico.

Cada operación se dibuja como la **herramienta de posición de
TradingView**: dos rectángulos apilados sobre la línea de entrada.

- **rectángulo verde** — de la entrada al objetivo más lejano que
  llegó a tocarse. Si la operación murió antes del primer objetivo no
  hay verde, solo rojo
- **rectángulo rojo** — de la entrada al stop
- **línea oscura** — el nivel del FRVP donde entró la orden

En un **long** el verde queda arriba y el rojo abajo; en un **short**,
al revés. La dirección se lee de un vistazo, sin mirar el marcador.

El triángulo de la entrada (**▲ long**, **▼ short**) va verde si la
operación terminó en positivo y rojo si no. Al pasar el ratón se ve el
nivel, el stop, el motivo de salida y el resultado en R.

Las tres vistas de operaciones son **"Solo principales" con las
posiciones encima**: las líneas del FRVP siguen dibujadas, que es lo
que hace falta para juzgar si la entrada tenía sentido.

Las operaciones duran una mediana de 8 horas, o sea dos velas: sobre
dos años de gráfico son invisibles. Por eso la caja tiene un ancho
mínimo de medio día, para poder localizarlas antes de hacer zoom.

De fondo, en punteado muy tenue, **la rejilla completa de niveles
operables**: todos los VAH/POC/VAL vigentes de los rangos
principales, cada uno desde el `confirmado_en` de su rango. Es la
rejilla que usó el backtest, así que explica de dónde sale cada
entrada y dónde estaban los objetivos.

Ojo, esa rejilla usa la **selección causal**
(`core.levels.seleccionar_causalmente`) y no la del resto del
gráfico: conserva los rangos que fueron el elegido de su zona en su
momento, aunque después apareciera otro solapado con mejor nota. Por
eso hay más líneas punteadas que niveles de color.

In [6]:
DIRECTORIO_TRADES = PROJECT_ROOT / "experiments" / "resultados"

# Verde y rojo apagados, para que convivan con el fondo arena sin
# chillar: el gráfico ya lleva cinco colores de ventana.
VERDE_OP = "#2f6b45"   # zona de objetivo
ROJO_OP = "#a33b2a"    # zona de riesgo
GRIS_OP = "#33291d"    # línea de entrada

# Opacidades de las cajas de posición. Bajas a propósito: se dibujan
# ENCIMA de las velas y del FRVP, y tienen que dejarlos ver.
RELLENO_OP = 0.18
BORDE_OP = 0.40

# Ancho mínimo de una caja de posición. La duración mediana es de 8 h
# (dos velas), invisible sobre dos años de histórico.
ANCHO_MINIMO_OP = pd.Timedelta("12h")

GROSOR_OP = 1.0
TAMANO_MARCA_OP = 7


def cargar_trades(symbol):
    """Lee del CSV las operaciones del experimento de toques del FRVP.

    Parameters
    ----------
    symbol : str
        Símbolo unificado de CCXT.

    Returns
    -------
    pd.DataFrame | None
        Las operaciones, o None si el experimento no se ha corrido
        todavía para ese símbolo.
    """
    nombre = symbol.replace("/", "-").replace(":", "-")
    ruta = DIRECTORIO_TRADES / f"trades_{nombre}.csv"
    if not ruta.exists():
        print(f"  (sin operaciones para {symbol}: falta {ruta.name})")
        return None

    trades = pd.read_csv(ruta)
    for columna in ("ts_entrada", "ts_salida"):
        trades[columna] = pd.to_datetime(trades[columna], utc=True)
    return trades


def precio_ultimo_tp(trades):
    """Precio del objetivo más lejano que llegó a tocarse.

    Parameters
    ----------
    trades : pd.DataFrame
        Operaciones cargadas con `cargar_trades`.

    Returns
    -------
    np.ndarray
        Precio del último TP alcanzado, o NaN si la operación murió
        antes de alcanzar el primero.
    """
    alcanzados = trades["objetivos_alcanzados"].to_numpy()
    precios = np.full(len(trades), np.nan)
    for k in (1, 2, 3):
        marca = alcanzados == k
        precios[marca] = trades.loc[marca, f"tp{k}"].to_numpy()
    return precios


def segmentos_horizontales(x0, x1, y):
    """Encadena muchas líneas horizontales en UNA sola traza.

    Plotly redibuja cada `shape` por separado, así que centenares de
    ellas ralentizan el pan y el zoom. Metidas en una traza con un
    `None` de separación entre segmentos, se dibujan de una vez.

    Parameters
    ----------
    x0, x1 : array-like
        Extremos izquierdo y derecho de cada segmento.
    y : array-like
        Altura de cada segmento. Los NaN se omiten.

    Returns
    -------
    tuple[list, list]
        Coordenadas x e y, con `None` entre segmentos.
    """
    xs, ys = [], []
    for a, b, altura in zip(x0, x1, y):
        if altura is None or not np.isfinite(altura):
            continue
        xs += [a, b, None]
        ys += [altura, altura, None]
    return xs, ys


def poligonos_caja(x0, x1, y0, y1):
    """Encadena muchos rectángulos en UNA sola traza.

    Mismo motivo que en `segmentos_horizontales`: con 180-260
    operaciones, otras tantas `shapes` de Plotly harían el zoom
    inutilizable. Con `fill="toself"`, cada tramo separado por `None`
    se rellena como un polígono independiente.

    Parameters
    ----------
    x0, x1 : array-like
        Fechas de apertura y cierre de cada caja.
    y0, y1 : array-like
        Precio de la base y del techo de cada caja. Los NaN se omiten:
        una operación que no llegó a tocar ningún objetivo no tiene
        caja verde.

    Returns
    -------
    tuple[list, list]
        Coordenadas x e y, con `None` entre rectángulos.
    """
    xs, ys = [], []
    for a, b, base, techo in zip(x0, x1, y0, y1):
        if not np.isfinite(base) or not np.isfinite(techo):
            continue
        xs += [a, b, b, a, a, None]
        ys += [base, base, techo, techo, base, None]
    return xs, ys


def trazas_grupo(t):
    """Las cuatro trazas que dibujan un grupo de operaciones.

    Cada operación se dibuja como la herramienta de posición de
    TradingView: dos rectángulos apilados sobre la línea de entrada,
    el verde hacia el objetivo y el rojo hacia el stop. En un long el
    verde queda arriba; en un short, abajo. Así la dirección se lee
    sin necesidad de mirar el marcador.

    Parameters
    ----------
    t : pd.DataFrame
        Subconjunto de operaciones (ganadoras o perdedoras).

    Returns
    -------
    list[go.Scatter]
        Caja de objetivo, caja de riesgo, línea de entrada y marcas de
        entrada, en ese orden.
    """
    # Una caja tan estrecha como la operación real sería invisible: se
    # le da un ancho mínimo sin mover la fecha de entrada.
    corta = (t["ts_salida"] - t["ts_entrada"]) < ANCHO_MINIMO_OP
    x1 = t["ts_salida"].where(~corta, t["ts_entrada"] + ANCHO_MINIMO_OP)
    x0 = t["ts_entrada"]

    entrada = t["entrada"].to_numpy()
    objetivo = precio_ultimo_tp(t)
    stop = t["stop_inicial"].to_numpy()

    def caja(otro_lado, color):
        xs, ys = poligonos_caja(x0, x1, entrada, otro_lado)
        return go.Scatter(
            x=xs, y=ys, mode="lines", fill="toself",
            fillcolor=rgba(color, RELLENO_OP),
            line={"color": rgba(color, BORDE_OP), "width": 0.7},
            hoverinfo="skip", showlegend=False, visible=False,
        )

    xs, ys = segmentos_horizontales(x0, x1, entrada)
    linea_entrada = go.Scatter(
        x=xs, y=ys, mode="lines",
        line={"color": rgba(GRIS_OP, 0.70), "width": GROSOR_OP},
        hoverinfo="skip", showlegend=False, visible=False,
    )

    texto = [
        f"{fila.direccion} {fila.nivel.upper()} · entrada {fila.entrada:,.4f}"
        f" · SL {fila.stop_inicial:,.4f}"
        f" · {fila.motivo_salida} · {fila.pnl_r:+.2f} R"
        for fila in t.itertuples()
    ]
    marcas = go.Scatter(
        x=x0, y=t["entrada"], mode="markers",
        marker={
            "symbol": [
                "triangle-up" if d == "long" else "triangle-down"
                for d in t["direccion"]
            ],
            "size": TAMANO_MARCA_OP,
            "color": [
                rgba(VERDE_OP if p > 0 else ROJO_OP, 0.85)
                for p in t["pnl_pct"]
            ],
            "line": {"width": 0},
        },
        text=texto, hovertemplate="%{text}<extra></extra>",
        showlegend=False, visible=False,
    )

    return [caja(objetivo, VERDE_OP), caja(stop, ROJO_OP), linea_entrada, marcas]


def trazas_operaciones(trades, niveles):
    """Monta las trazas de la capa de operaciones: 4 por grupo.

    No dibuja ninguna línea de nivel propia. Con
    `seleccion_rangos: "global"` la rejilla que opera el backtest es
    EXACTAMENTE la que ya pinta el FRVP de los rangos principales, así
    que añadir otra capa de líneas sería duplicarlas.

    Se comprueba esa correspondencia y se avisa si falla: una entrada
    sin su línea debajo significaría que el gráfico y el backtest han
    dejado de mirar los mismos niveles.

    Parameters
    ----------
    trades : pd.DataFrame | None
        Operaciones del experimento, o None si no las hay.
    niveles : pd.DataFrame
        Rejilla de niveles operables, para la comprobación.

    Returns
    -------
    tuple[list[go.Scatter], dict[str, list[bool]]]
        Las trazas y, para cada modo de vista ("todas", "ganadoras",
        "perdedoras"), qué trazas se ven.
    """
    if trades is None or trades.empty or niveles.empty:
        return [], {}

    huerfanas = int((~np.isin(
        np.round(trades["entrada"].to_numpy(), 6),
        np.round(niveles["precio"].to_numpy(), 6),
    )).sum())
    if huerfanas:
        print(f"  AVISO: {huerfanas} entradas no caen sobre ningún nivel dibujado")

    ganadoras = trades[trades["pnl_pct"] > 0]
    perdedoras = trades[trades["pnl_pct"] <= 0]
    trazas = trazas_grupo(ganadoras) + trazas_grupo(perdedoras)

    return trazas, {
        "todas": [True] * 4 + [True] * 4,
        "ganadoras": [True] * 4 + [False] * 4,
        "perdedoras": [False] * 4 + [True] * 4,
    }

## Imbalances semanales

Franjas de precio que el mercado atravesó sin negociar, detectadas en
el **semanal** con el patrón de tres velas (SPEC.md §9): si el mínimo
de la tercera queda por encima del máximo de la primera, ese hueco no
se negoció.

Se dibujan como **rectángulos morados** que arrancan en la vela que
los confirma y se estiran hacia la derecha, igual que los niveles del
FRVP.

**Se van comiendo.** Cuando el precio vuelve a entrar en la franja, la
parte visitada deja de ser hueco y el rectángulo se estrecha: por eso
tienen forma de escalera y no de rectángulo liso. Lo que queda sin
visitar sigue vivo aunque sea un 10% del original. Cuando el precio lo
recorre entero, el imbalance muere y el dibujo termina ahí.

El relleno se mide **con mechas**: basta con que el precio pase por la
zona, no hace falta que cierre dentro.

Los que llegan hasta el borde derecho del gráfico son los que siguen
**sin rellenar hoy**, y son los candidatos a zona de objetivo.

In [7]:
MORADO_IMB = "#7a5aa8"

# Relleno bajo: los imbalances son zonas de fondo, no deben tapar el
# precio ni competir con las cajas de posición.
RELLENO_IMB = 0.16
BORDE_IMB = 0.45


def poligono_escalonado(instantes, borde, fijo, hacia_arriba):
    """Contorno de un imbalance que se va comiendo con el tiempo.

    El borde móvil solo avanza en una dirección (un imbalance nunca se
    "descome"), así que el contorno es una escalera. Se emiten vértices
    únicamente donde el borde CAMBIA: con velas de 4h sobre dos años
    serían miles de puntos por zona, y en realidad son unas pocas
    decenas de escalones.

    Parameters
    ----------
    instantes : pd.DatetimeIndex
        Velas evaluadas, de la confirmación en adelante.
    borde : np.ndarray
        Posición del borde móvil en cada instante.
    fijo : float
        El lado que no se mueve (el suelo en un imbalance alcista, el
        techo en uno bajista).
    hacia_arriba : bool
        True si el lado móvil es el inferior (imbalance bajista).

    Returns
    -------
    tuple[list, list]
        Coordenadas x e y del polígono cerrado.
    """
    if len(instantes) == 0:
        return [], []

    cambios = np.flatnonzero(np.diff(borde) != 0.0) + 1
    posiciones = np.concatenate([[0], cambios, [len(borde) - 1]])

    xs, ys = [], []
    anterior = borde[0]
    for pos in posiciones:
        if borde[pos] != anterior:
            # Escalón: primero se prolonga el nivel viejo hasta aquí.
            xs.append(instantes[pos])
            ys.append(anterior)
            anterior = borde[pos]
        xs.append(instantes[pos])
        ys.append(borde[pos])

    # Se cierra por el lado fijo, de vuelta hacia el inicio.
    xs += [instantes[-1], instantes[0]]
    ys += [fijo, fijo]
    return xs, ys


def trazas_imbalances(symbol):
    """Dibuja los imbalances semanales, cada uno hasta que se rellena.

    Parameters
    ----------
    symbol : str
        Símbolo unificado de CCXT.

    Returns
    -------
    tuple[list[go.Scatter], int, int]
        Las trazas, cuántos imbalances hay en total y cuántos siguen
        vivos al final del histórico.
    """
    imbalances = detectar_imbalances(velas_semanales[symbol])
    if imbalances.empty:
        return [], 0, 0

    df = velas[symbol]
    trazas, vivos = [], 0

    for fila in imbalances.to_dict("records"):
        instantes, borde, muerto_en = evolucion_imbalance(pd.Series(fila), df)
        if len(instantes) == 0:
            continue

        # El dibujo termina donde el imbalance se rellena del todo.
        if muerto_en is not None:
            hasta = instantes.searchsorted(muerto_en, side="right")
            instantes, borde = instantes[:hasta], borde[:hasta]
        else:
            vivos += 1

        alcista = fila["tipo"] == "alcista"
        fijo = fila["suelo"] if alcista else fila["techo"]
        xs, ys = poligono_escalonado(instantes, borde, fijo, not alcista)
        if not xs:
            continue

        estado = "sin rellenar" if muerto_en is None else "rellenado"
        trazas.append(go.Scatter(
            x=xs, y=ys, mode="lines", fill="toself",
            fillcolor=rgba(MORADO_IMB, RELLENO_IMB),
            line={"color": rgba(MORADO_IMB, BORDE_IMB), "width": 0.7},
            text=(f"imbalance {fila['tipo']} 1w · "
                  f"{fila['suelo']:,.4f}–{fila['techo']:,.4f} · {estado}"),
            hovertemplate="%{text}<extra></extra>",
            showlegend=False, visible=False,
        ))

    return trazas, len(imbalances), vivos

## Panel de régimen: SQZ + ADX + TTM

El panel de abajo reproduce el indicador **SQZ+ADX+TTM** tal como lo
mira el autor en TradingView, con su misma configuración (BB 20/2.0,
KC 20/1.5, momento lineal 20, ADX 14). Sirve para contrastar de un
vistazo lo que ve el bot con lo que ve él.

**Histograma del TTM**, en los cuatro colores clásicos:

| color | histograma | va | significa |
|---|---|---|---|
| verde oscuro | positivo | subiendo | impulso alcista acelerando |
| verde claro | positivo | bajando | impulso alcista agotándose |
| rojo oscuro | negativo | bajando | impulso bajista acelerando |
| rojo claro | negativo | subiendo | impulso bajista agotándose |

**Línea del ADX** con su Key Level en 23, más el umbral de 35 que usa
el bot como filtro: por encima de ese nivel no se abre ninguna
operación, porque la tendencia es demasiado fuerte para una reversión.

**Puntos bajo el eje** cuando hay *squeeze* (Bollinger dentro de
Keltner): volatilidad comprimida, y otro estado en el que el bot no
opera.

Ojo con una diferencia importante entre lo que ves y lo que hace el
bot: la lectura manual habitual compra en «valle rojo desarrollado»
—el histograma agotándose—, y **medido sobre estas operaciones eso
rinde peor** (-0.086 R frente a +0.401 cuando el histograma acelera).
El bot puntúa lo contrario. Está en SPEC.md §17 con los números.

In [8]:
# Los cuatro colores del histograma del TTM. Oscuro = acelerando,
# claro = agotándose, que es la convención del indicador original.
COLOR_FASE = {
    "alcista_fuerte": "#2f6b45",   # verde oscuro
    "alcista_debil": "#8fbf9f",    # verde claro
    "bajista_fuerte": "#a33b2a",   # rojo oscuro
    "bajista_debil": "#d9a49b",    # rojo claro
    "": "#c9bfae",                 # sin dato
}

AZUL_ADX = "#3b5b8c"
MORADO_SQZ = "#7a5aa8"


def trazas_regimen(symbol):
    """Panel inferior con el histograma del TTM, el ADX y el squeeze.

    Reproduce el indicador SQZ+ADX+TTM con la misma configuración que
    usa el autor a mano, para poder comparar el gráfico del bot con el
    de TradingView.

    Parameters
    ----------
    symbol : str
        Símbolo unificado de CCXT.

    Returns
    -------
    list[go.Scatter | go.Bar]
        Trazas para la fila inferior de la figura.
    """
    df = velas[symbol]
    cfg = config["experimento_toques_frvp"]

    periodo = int(cfg.get("squeeze_periodo", 20))
    momento = momento_ttm(df, periodo)
    fases = fase_ttm(momento)
    tabla_adx = adx(df, int(cfg.get("adx_periodo", 14)))
    comprimido = squeeze(
        df, periodo,
        float(cfg.get("squeeze_desviaciones", 2.0)),
        float(cfg.get("squeeze_multiplicador_keltner", 1.5)),
    )["activo"].fillna(False).astype(bool)

    # El histograma y el ADX viven en escalas muy distintas (el momento
    # es una pendiente en unidades de precio y el ADX va de 0 a 100), así
    # que el histograma se normaliza para que quepan en el mismo panel.
    escala = momento.abs().rolling(500, min_periods=50).quantile(0.95)
    normalizado = (momento / escala.replace(0, np.nan) * 50.0).clip(-100, 100)

    barras = go.Bar(
        x=df.index, y=normalizado,
        marker={
            "color": [COLOR_FASE.get(f, COLOR_FASE[""]) for f in fases],
            "line": {"width": 0},
        },
        name="TTM",
        hovertemplate="TTM %{y:.0f} · %{text}<extra></extra>",
        text=fases,
        showlegend=False, visible=False,
    )

    linea_adx = go.Scatter(
        x=df.index, y=tabla_adx["adx"],
        mode="lines", line={"color": AZUL_ADX, "width": 1.2},
        name="ADX",
        hovertemplate="ADX %{y:.1f}<extra></extra>",
        showlegend=False, visible=False,
    )

    # Marcas de squeeze bajo el eje, donde no estorban al histograma.
    instantes = df.index[comprimido]
    puntos_squeeze = go.Scatter(
        x=instantes, y=np.full(len(instantes), -95.0),
        mode="markers",
        marker={"color": rgba(MORADO_SQZ, 0.8), "size": 3, "symbol": "square"},
        name="squeeze",
        hovertemplate="squeeze: volatilidad comprimida<extra></extra>",
        showlegend=False, visible=False,
    )

    return [barras, linea_adx, puntos_squeeze]


def formas_regimen(x_inicio, x_fin):
    """Líneas de referencia del panel: Key Level y umbral del filtro.

    Parameters
    ----------
    x_inicio, x_fin : pd.Timestamp
        Extremos del eje temporal.

    Returns
    -------
    list[dict]
        Formas de Plotly ancladas al eje del panel inferior.
    """
    cfg = config["experimento_toques_frvp"]
    umbral = cfg.get("adx_maximo")

    formas = [
        # Cero del histograma.
        {
            "type": "line", "xref": "x3", "yref": "y3",
            "x0": x_inicio, "x1": x_fin, "y0": 0, "y1": 0,
            "line": {"color": rgba(TEXTO_TENUE, 0.5), "width": 0.8},
        },
        # Key Level 23: el que usa el autor en su indicador.
        {
            "type": "line", "xref": "x3", "yref": "y3",
            "x0": x_inicio, "x1": x_fin, "y0": 23, "y1": 23,
            "line": {"color": rgba(AZUL_ADX, 0.4), "width": 0.8, "dash": "dot"},
        },
    ]
    if umbral is not None:
        # Umbral del filtro: por encima, el bot no abre nada.
        formas.append({
            "type": "line", "xref": "x3", "yref": "y3",
            "x0": x_inicio, "x1": x_fin, "y0": umbral, "y1": umbral,
            "line": {"color": rgba("#a33b2a", 0.55), "width": 1.0, "dash": "dash"},
        })
    return formas

## Construcción de la figura

Las cinco vistas del menú se calculan por adelantado: cada botón
lleva su propia lista de formas y decide qué perfiles se ven. Los
perfiles se crean UNA vez y se muestran u ocultan, no se duplican
por vista.

In [9]:
# Vistas del menú desplegable. Cada una filtra los rangos y decide si
# se dibujan los perfiles, los imbalances y qué operaciones se ven.
#   (nombre, conjunto, tipo, con_frvp, imbalances, operaciones)
# `operaciones` es None, "todas", "ganadoras" o "perdedoras". Las tres
# vistas de operaciones son "Solo principales" con las posiciones
# dibujadas encima: las líneas del FRVP son justo lo que hay que ver
# para juzgar si cada entrada tenía sentido.
VISTAS = [
    ("Completa", "seleccionados", None, True, False, None),
    ("Solo principales", "seleccionados", "principal", True, False, None),
    ("Solo secundarios", "seleccionados", "secundario", True, False, None),
    ("Sin FRVP", "seleccionados", None, False, False, None),
    ("Todos los detectados", "crudos", None, False, False, None),
    ("Imbalances 1w", "seleccionados", "principal", True, True, None),
    ("Operaciones", "seleccionados", "principal", True, True, "todas"),
    ("Ops ganadoras", "seleccionados", "principal", True, True, "ganadoras"),
    ("Ops perdedoras", "seleccionados", "principal", True, True, "perdedoras"),
]


def construir_capas(symbol):
    """Calcula, para cada vista, sus formas y qué trazas se ven.

    Los perfiles son trazas y se crean solo para los rangos
    seleccionados: en la vista de todos los detectados hay más de un
    centenar de rangos, y calcularles el FRVP dispararía el peso del
    HTML sin que se distinga nada en pantalla.

    El orden de las trazas es fijo y lo aprovechan los botones del
    menú: primero los perfiles, después los imbalances y al final las
    operaciones.

    Parameters
    ----------
    symbol : str
        Símbolo unificado de CCXT.

    Returns
    -------
    tuple[list, list, dict, pd.DataFrame]
        Trazas, sus rangos asociados, las capas por vista y los rangos
        seleccionados.
    """
    df = velas[symbol]
    inicio = pd.Timestamp(FECHA_INICIO, tz="UTC") if FECHA_INICIO else df.index.min()
    fin = pd.Timestamp(FECHA_FIN, tz="UTC") if FECHA_FIN else df.index.max()

    crudos = detectar_rangos_laterales(df, config)
    seleccionados = seleccionar_rangos(crudos, config)

    def visibles(r):
        return r[(r["fin"] >= inicio) & (r["inicio"] <= fin)]

    sel = visibles(seleccionados)
    todos = visibles(crudos)

    # Perfiles: una traza por rango seleccionado, creada una sola vez.
    trazas, rangos_traza, perfiles = [], [], {}
    for i, rango in enumerate(sel.itertuples()):
        color = COLOR_POR_VENTANA.get(rango.ventana, COLOR_POR_DEFECTO)
        vis = visibilidad(rango.calidad)
        perfil = perfil_de(symbol, rango)
        if perfil is None:
            continue
        traza = traza_perfil(rango, perfil, color, vis)
        if traza is None:
            continue
        perfiles[i] = perfil
        trazas.append(traza)
        rangos_traza.append((i, rango, color, vis))

    # Imbalances semanales, antes de las operaciones para que queden
    # por debajo de las cajas de posición.
    imbs, n_imb, n_vivos = trazas_imbalances(symbol)
    trazas += imbs

    # Capa de operaciones. La rejilla se construye a partir de los
    # rangos CRUDOS: `construir_niveles` aplica por dentro el modo de
    # selección que diga la configuración.
    trades = cargar_trades(symbol)
    niveles = construir_niveles(crudos, df, velas_finas[symbol], config)
    ops, visibles_ops = trazas_operaciones(trades, niveles)
    trazas += ops

    capas = {}
    for nombre, conjunto, tipo, con_frvp, con_imb, modo_ops in VISTAS:
        if modo_ops is not None and not ops:
            continue
        if con_imb and not imbs:
            continue

        datos = sel if conjunto == "seleccionados" else todos
        if tipo is not None:
            datos = datos[datos["tipo"] == tipo]

        formas, etiquetas, indices_visibles = [], [], set()
        for i, rango in enumerate(datos.itertuples()):
            color = COLOR_POR_VENTANA.get(rango.ventana, COLOR_POR_DEFECTO)
            vis = visibilidad(rango.calidad)
            formas.append(forma_caja(rango, color, vis))
            # En la vista de todos los detectados hay más de un
            # centenar de rangos: rotularlos todos taparía el precio.
            if conjunto == "seleccionados":
                etiquetas.append(etiqueta_rango(rango, color, vis))

        if con_frvp:
            claves = set(zip(datos["inicio"], datos["ventana"]))
            for i, rango, color, vis in rangos_traza:
                if (rango.inicio, rango.ventana) not in claves:
                    continue
                indices_visibles.add(i)
                # Los niveles se proyectan a la derecha solo en los
                # principales: cada línea extendida es una zona a
                # vigilar, y de más se vuelven ilegibles.
                x_fin = fin if rango.tipo == "principal" else rango.fin
                formas += formas_niveles(rango, perfiles[i], color, vis, x_fin)

        ver_ops = (
            visibles_ops[modo_ops] if modo_ops is not None
            else [False] * len(ops)
        )

        if modo_ops == "todas":
            cuenta = f"{len(trades)} ops"
        elif modo_ops == "ganadoras":
            cuenta = f"{int((trades['pnl_pct'] > 0).sum())} ops"
        elif modo_ops == "perdedoras":
            cuenta = f"{int((trades['pnl_pct'] <= 0).sum())} ops"
        elif con_imb:
            cuenta = f"{n_vivos} sin rellenar de {n_imb}"
        else:
            cuenta = f"{len(datos)} rangos"

        capas[nombre] = {
            "shapes": formas,
            "annotations": etiquetas,
            "visible": (
                [i in indices_visibles for i, _, _, _ in rangos_traza]
                + [con_imb] * len(imbs)
                + ver_ops
            ),
            "cuenta": cuenta,
            "n_cajas": len(datos),
        }

    return trazas, rangos_traza, capas, sel


def construir_figura(symbol):
    """Dibuja las velas de 4h de un símbolo con sus rangos y perfiles.

    La detección se corre sobre TODO el histórico en caché, no solo
    sobre el intervalo visualizado, para no perder rangos cuyo tramo
    empiece antes de FECHA_INICIO pero siga vigente dentro de la
    ventana visible.

    Parameters
    ----------
    symbol : str
        Símbolo unificado de CCXT.

    Returns
    -------
    tuple[go.Figure, pd.DataFrame, dict]
        La figura, los rangos seleccionados y las capas por vista.
    """
    df = velas[symbol]
    inicio = pd.Timestamp(FECHA_INICIO, tz="UTC") if FECHA_INICIO else df.index.min()
    fin = pd.Timestamp(FECHA_FIN, tz="UTC") if FECHA_FIN else df.index.max()
    df_rango = df.loc[inicio:fin]

    trazas, _, capas, sel = construir_capas(symbol)

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.66, 0.12, 0.22], vertical_spacing=0.02,
    )
    # Velas monocromas: huecas al alza, rellenas a la baja.
    fig.add_trace(
        go.Candlestick(
            x=df_rango.index,
            open=df_rango["open"], high=df_rango["high"],
            low=df_rango["low"], close=df_rango["close"],
            name=symbol,
            increasing={"line": {"color": VELA, "width": GROSOR_VELA},
                        "fillcolor": FONDO},
            decreasing={"line": {"color": VELA, "width": GROSOR_VELA},
                        "fillcolor": VELA},
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Bar(
            x=df_rango.index, y=df_rango["volume"], name="Volumen",
            marker={"color": rgba(VELA, 0.20), "line": {"width": 0}},
        ),
        row=2, col=1,
    )
    for traza in trazas:
        fig.add_trace(traza, row=1, col=1)

    # Panel de régimen (SQZ + ADX + TTM), en la fila de abajo.
    regimen = trazas_regimen(symbol)
    for traza in regimen:
        traza.visible = True
        fig.add_trace(traza, row=3, col=1)

    # Velas, volumen y las tres trazas del panel están siempre
    # visibles; los botones solo gobiernan perfiles, imbalances y
    # operaciones. El panel va al final para no descuadrar los índices
    # que usan esos botones.
    fijas = [True, True]
    inicial = next(iter(capas))

    # Las líneas de referencia del panel (cero, Key Level 23 y umbral
    # del filtro) acompañan a las formas de cada vista.
    referencias = formas_regimen(inicio, fin)
    for capa in capas.values():
        capa["shapes"] = list(capa["shapes"]) + referencias

    fig.update_layout(
        shapes=capas[inicial]["shapes"],
        annotations=capas[inicial]["annotations"],
        title={
            "text": f"{symbol}   ·   4h   ·   rangos del Filtro 1 y su FRVP",
            "font": {"color": TEXTO_TENUE, "size": 12},
            "x": 0.5, "xanchor": "center", "y": 0.985,
        },
        xaxis_rangeslider_visible=False,
        height=840,
        showlegend=False,
        paper_bgcolor=FONDO,
        plot_bgcolor=FONDO,
        font={"color": TEXTO, "size": 11, "family": FUENTE},
        margin={"l": 16, "r": 78, "t": 84, "b": 44},
        hovermode="x unified",
        hoverlabel={
            "bgcolor": PANEL, "bordercolor": REJILLA,
            "font": {"color": TEXTO, "size": 11, "family": FUENTE},
        },
        updatemenus=[{
            "type": "dropdown",
            "direction": "down",
            "x": 0.0, "xanchor": "left",
            "y": 1.10, "yanchor": "top",
            "bgcolor": PANEL,
            "bordercolor": REJILLA,
            "font": {"color": TEXTO, "size": 11, "family": FUENTE},
            "pad": {"l": 8, "r": 8, "t": 4, "b": 4},
            "showactive": True,
            "buttons": [
                {
                    "label": f"{nombre}  ({capa['cuenta']})",
                    "method": "update",
                    "args": [
                        {"visible": fijas + capa["visible"]
                         + [True] * len(regimen)},
                        {
                            "shapes": capa["shapes"],
                            "annotations": capa["annotations"],
                        },
                    ],
                }
                for nombre, capa in capas.items()
            ],
        }],
    )
    fig.data[0].visible = True
    fig.data[1].visible = True
    for traza, ver in zip(fig.data[2:], capas[inicial]["visible"]):
        traza.visible = ver
    for traza in regimen:
        traza.visible = True

    # Ejes al estilo TradingView: precio a la derecha, sin líneas de eje,
    # rejilla horizontal tenue y crosshair al pasar el ratón.
    fig.update_xaxes(
        showgrid=False, showline=False, zeroline=False,
        ticks="outside", ticklen=4, tickcolor=REJILLA,
        showspikes=True, spikemode="across", spikesnap="cursor",
        spikecolor=TEXTO_TENUE, spikethickness=1, spikedash="dot",
    )
    # Sin rejilla horizontal: se confundia con las lineas del FRVP,
    # que son horizontales y se proyectan a lo ancho del grafico. Las
    # marcas del eje de precio bastan para situarse.
    fig.update_yaxes(
        side="right", showgrid=False,
        showline=False, zeroline=False,
        ticks="outside", ticklen=4, tickcolor=REJILLA,
        showspikes=True, spikemode="across", spikesnap="cursor",
        spikecolor=TEXTO_TENUE, spikethickness=1, spikedash="dot",
    )
    # El volumen es contexto, no dato principal: sin rejilla y con menos
    # marcas para que no compita con el precio.
    fig.update_yaxes(showgrid=False, nticks=3, row=2, col=1)
    # El panel de régimen mezcla dos escalas ya normalizadas a 0-100,
    # así que se fija el rango para que el histograma no lo reescale.
    fig.update_yaxes(
        showgrid=False, nticks=4, range=[-100, 100], row=3, col=1,
    )

    return fig, sel, capas

## Gráficos y exportación

Cada figura se guarda además como HTML autocontenido en `notebooks/`.
`include_plotlyjs=True` incrusta la librería en el propio fichero:
pesa unos MB, pero se abre en el navegador sin Jupyter, sin conexión y
sin depender de un CDN.

In [10]:
def nombre_fichero(symbol):
    """Deriva un nombre de fichero seguro del símbolo unificado de CCXT.

    Parameters
    ----------
    symbol : str
        Símbolo unificado de CCXT (p. ej. "ONDO/USD:USD").

    Returns
    -------
    str
        Nombre en minúsculas, p. ej. "rangos_ondo.html".
    """
    return f"rangos_{symbol.split('/')[0].lower()}.html"


for symbol in SIMBOLOS:
    fig, sel, capas = construir_figura(symbol)

    print(f"\n{'=' * 78}")
    print(f"{symbol}")
    print("=" * 78)
    print("\nVistas del menú:")
    for nombre, capa in capas.items():
        print(f"  {nombre:>22}: {capa['cuenta']}")

    v = sel.assign(dias=(sel["fin"] - sel["inicio"]).dt.days)
    print()
    print(v.groupby("tipo").agg(
        n=("inicio", "size"),
        dias_mediana=("dias", "median"),
        calidad_mediana=("calidad", "median"),
    ).round(2).to_string())

    principales = v[v["tipo"] == "principal"].sort_values("inicio")
    if not principales.empty:
        print("\nRANGOS PRINCIPALES (los más operables):")
        print(principales[[
            "inicio", "fin", "dias", "suelo", "techo", "calidad", "ventana",
        ]].to_string(index=False))

    # Resumen de las operaciones dibujadas, para poder contrastar el
    # gráfico con el informe de `experiments/exp_toques_frvp.py`.
    trades = cargar_trades(symbol)
    if trades is not None and not trades.empty:
        print("\nOPERACIONES DEL EXPERIMENTO (toques del FRVP):")
        print(trades.groupby(["nivel", "direccion"]).agg(
            ops=("pnl_r", "size"),
            acierto=("pnl_pct", lambda s: (s > 0).mean()),
            r_medio=("pnl_r", "mean"),
        ).round(3).to_string())

    ruta_html = PROJECT_ROOT / "notebooks" / nombre_fichero(symbol)
    try:
        fig.write_html(ruta_html, include_plotlyjs=True, full_html=True)
    except OSError:
        print(f"No se pudo escribir el HTML en {ruta_html}")
        raise
    print(f"\nHTML: {ruta_html.name} "
          f"({ruta_html.stat().st_size / 1024 / 1024:.1f} MB)")

    fig.show()


ONDO/USD:USD

Vistas del menú:
                Completa: 27 rangos
        Solo principales: 10 rangos
        Solo secundarios: 17 rangos
                Sin FRVP: 27 rangos
    Todos los detectados: 101 rangos
           Imbalances 1w: 6 sin rellenar de 18
             Operaciones: 89 ops
           Ops ganadoras: 30 ops
          Ops perdedoras: 59 ops

             n  dias_mediana  calidad_mediana
tipo                                         
principal   10          50.0             0.75
secundario  17          13.0             0.75

RANGOS PRINCIPALES (los más operables):
                   inicio                       fin  dias  suelo  techo  calidad  ventana
2024-10-01 12:00:00+00:00 2024-11-10 16:00:00+00:00    40 0.6043 0.8345    0.814      250
2024-12-02 20:00:00+00:00 2024-12-27 00:00:00+00:00    24 1.4670 2.0173    0.515      150
2024-12-26 00:00:00+00:00 2025-02-24 04:00:00+00:00    60 1.0924 1.6061    0.837      250
2025-03-09 08:00:00+00:00 2025-04-06 12:00:00+00:00    


HTML: rangos_ondo.html (5.4 MB)



BTC/USD:USD

Vistas del menú:
                Completa: 20 rangos
        Solo principales: 9 rangos
        Solo secundarios: 11 rangos
                Sin FRVP: 20 rangos
    Todos los detectados: 116 rangos
           Imbalances 1w: 4 sin rellenar de 18
             Operaciones: 68 ops
           Ops ganadoras: 21 ops
          Ops perdedoras: 47 ops

             n  dias_mediana  calidad_mediana
tipo                                         
principal    9          72.0             0.77
secundario  11           9.0             0.74

RANGOS PRINCIPALES (los más operables):
                   inicio                       fin  dias    suelo    techo  calidad  ventana
2024-09-19 00:00:00+00:00 2024-10-15 08:00:00+00:00    26  59433.0  65874.0    0.703      150
2024-11-12 16:00:00+00:00 2025-02-26 12:00:00+00:00   105  87392.0 106934.0    0.787      400
2025-02-25 20:00:00+00:00 2025-04-06 16:00:00+00:00    39  78569.0  96632.0    0.419      250
2025-05-09 08:00:00+00:00 2025-07-10 12:0


HTML: rangos_btc.html (5.3 MB)
